In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================================
# CELL 1 : IMPORTS, PATHS & HELPER TO LOAD IMAGES
# ============================================================================
import os, time, warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import cv2
from scipy import stats
import pywt
from skimage.feature import graycomatrix

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# ---------- EDIT THIS PATH ----------
CROPPED_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/cropped_leaves'
OUTPUT_DIR  = '/kaggle/working/feature_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Class names (must match the subfolders you used earlier)
CLASSES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
class_to_idx = {c:i for i,c in enumerate(CLASSES)}

# Target image size (must match your cropped images)
IMG_SIZE = 256

print("Libraries loaded. Output directory:", OUTPUT_DIR)

Libraries loaded. Output directory: /kaggle/working/feature_analysis


In [3]:
# ============================================================================
# CELL 2 : FEATURE EXTRACTION HELPER FUNCTIONS
# ============================================================================

def compute_14_features(mat: np.ndarray) -> np.ndarray:
    """
    Compute 14 statistical features from a 2D array.
    Features: area, mean, std, energy, median, skewness, entropy,
              min, max, mean_abs_dev, kurtosis, range, rms, uniformity.
    """
    mat = mat.ravel().astype(np.float64)
    area = len(mat)
    mean_val = np.mean(mat)
    std_val = np.std(mat)
    energy = np.sum(mat**2)
    median_val = np.median(mat)
    skewness = stats.skew(mat)
    kurtosis = stats.kurtosis(mat)
    min_val = np.min(mat)
    max_val = np.max(mat)
    range_val = max_val - min_val
    mad = np.mean(np.abs(mat - mean_val))
    rms = np.sqrt(np.mean(mat**2))

    # Entropy and uniformity via histogram (256 bins)
    counts, _ = np.histogram(mat, bins=256, range=(min_val, max_val))
    # If all values are identical, set uniform probabilities
    if max_val == min_val:
        probs = np.ones(256) / 256
    else:
        probs = counts / area
    probs += 1e-12  # avoid log(0)
    entropy_val = -np.sum(probs * np.log2(probs))
    uniformity = np.sum(probs**2)

    return np.array([area, mean_val, std_val, energy, median_val,
                     skewness, entropy_val, min_val, max_val, mad,
                     kurtosis, range_val, rms, uniformity])

def extract_glcm_features(gray_img):
    """GLCM with 14 features per orientation → 4*14 = 56 features."""
    glcm = graycomatrix(gray_img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    feats = []
    for a in range(4):
        mat = glcm[:, :, 0, a]
        feats.append(compute_14_features(mat))
    return np.concatenate(feats)

def extract_gldm_features(gray_img):
    """GLDM with 14 features per orientation → 56 features."""
    rows, cols = gray_img.shape
    offsets = [(1,0), (1,1), (0,1), (-1,1)]  # 0°, 45°, 90°, 135°
    all_feats = []
    for dx, dy in offsets:
        # Create difference matrix (size 256 x 256)
        diff_mat = np.zeros((256, 256), dtype=np.float64)
        for i in range(rows):
            for j in range(cols):
                val = gray_img[i, j]
                ni, nj = i+dx, j+dy
                if 0 <= ni < rows and 0 <= nj < cols:
                    diff = abs(int(val) - int(gray_img[ni, nj]))
                    diff_mat[val, diff] += 1
        # Normalize
        total = diff_mat.sum()
        if total > 0:
            diff_mat /= total
        all_feats.append(compute_14_features(diff_mat))
    return np.concatenate(all_feats)

def extract_fft_features(gray_img):
    """FFT magnitude → 14 features."""
    f = np.fft.fft2(gray_img)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    return compute_14_features(mag)

def extract_dwt_features(gray_img, wavelet='db2'):
    """2-level DWT → 8 sub-bands → 8*14 = 112 features."""
    # Level 1
    LL1, (LH1, HL1, HH1) = pywt.dwt2(gray_img, wavelet)
    # Level 2 (on LL1)
    LL2, (LH2, HL2, HH2) = pywt.dwt2(LL1, wavelet)
    bands = [LL1, LH1, HL1, HH1, LL2, LH2, HL2, HH2]
    feats = [compute_14_features(b) for b in bands]
    return np.concatenate(feats)

def extract_all_features(rgb_img):
    """
    Given a 256x256 RGB image, compute the full 252-dimensional feature vector.
    """
    gray = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2GRAY)
    # Texture (direct from image) -> 14
    tex_feat = compute_14_features(gray)
    # GLCM -> 56
    glcm_feat = extract_glcm_features(gray)
    # GLDM -> 56
    gldm_feat = extract_gldm_features(gray)
    # FFT -> 14
    fft_feat = extract_fft_features(gray)
    # DWT -> 112
    dwt_feat = extract_dwt_features(gray)
    return np.concatenate([tex_feat, glcm_feat, gldm_feat, fft_feat, dwt_feat])

print("Feature extraction functions defined.")

Feature extraction functions defined.


In [4]:
# ============================================================================
# CELL 3 (UPDATED) : COMPUTE 252 FEATURES FROM SEGMENTED (OVERLAY) IMAGES
# ============================================================================

import os
import numpy as np
import pandas as pd
import cv2
from tqdm.notebook import tqdm

# ---------- Important: use OVERLAY_DIR, not CROPPED_DIR ----------
OVERLAY_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'   # the blended images with disease masks
OUTPUT_DIR  = '/kaggle/working/feature_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# Collect overlay image paths
image_paths = []
labels = []
for cls_name in CLASSES:
    for fname in os.listdir(OVERLAY_DIR):
        if fname.startswith(cls_name + '__'):
            image_paths.append(os.path.join(OVERLAY_DIR, fname))
            labels.append(class_to_idx[cls_name])

print(f"Found {len(image_paths)} overlay images.")

# Extract 252 features from overlay images
feature_list = []
for impath in tqdm(image_paths, desc="Extracting features from overlays"):
    img_bgr = cv2.imread(impath)
    if img_bgr is None:
        print(f"Warning: cannot read {impath}, filling with zeros")
        feature_list.append(np.zeros(252))
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    feat = extract_all_features(img_rgb)   # uses the same helper functions from Cell 2
    feature_list.append(feat)

X = np.array(feature_list)
y = np.array(labels)

# Build DataFrame with column names (as before)
col_names = (
    ['TEX_' + str(i) for i in range(14)] +
    ['GLCM_' + str(i) for i in range(56)] +
    ['GLDM_' + str(i) for i in range(56)] +
    ['FFT_' + str(i) for i in range(14)] +
    ['DWT_' + str(i) for i in range(112)]
)
df = pd.DataFrame(X, columns=col_names)
df['label'] = y
df['image_path'] = image_paths

csv_path = os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv')
df.to_csv(csv_path, index=False)
print(f"Feature matrix (from overlays) saved to {csv_path}, shape = {X.shape}")

Found 11268 overlay images.


Extracting features from overlays:   0%|          | 0/11268 [00:00<?, ?it/s]

Feature matrix (from overlays) saved to /kaggle/working/feature_analysis/features_252_from_overlays.csv, shape = (11268, 252)


In [7]:
# ============================================================================
# CELL 4 : PCA (→70) & KPCA (→65) on the 252 features
# ============================================================================
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

# Load data
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
X = df.drop(columns=['label', 'image_path']).values
y = df['label'].values

# Standardize features (important for PCA/KPCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ----- PCA (70 components) -----
pca = PCA(n_components=70, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance ratio (first 5): {pca.explained_variance_ratio_[:5]}")

df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(70)])
df_pca['label'] = y
pca_path = os.path.join(OUTPUT_DIR, 'reduced_pca_70.csv')
df_pca.to_csv(pca_path, index=False)
print(f"PCA-reduced features saved to {pca_path}")

# ----- KPCA (RBF kernel, 65 components) -----
kpca = KernelPCA(n_components=65, kernel='rbf', random_state=42, n_jobs=-1)
X_kpca = kpca.fit_transform(X_scaled)

df_kpca = pd.DataFrame(X_kpca, columns=[f'KPC{i+1}' for i in range(65)])
df_kpca['label'] = y
kpca_path = os.path.join(OUTPUT_DIR, 'reduced_kpca_65.csv')
df_kpca.to_csv(kpca_path, index=False)
print(f"KPCA-reduced features saved to {kpca_path}")

PCA explained variance ratio (first 5): [0.31632593 0.18111946 0.13558606 0.08135131 0.03490979]
PCA-reduced features saved to /kaggle/working/feature_analysis/reduced_pca_70.csv
KPCA-reduced features saved to /kaggle/working/feature_analysis/reduced_kpca_65.csv


In [9]:
# ============================================================================
# CELL 5 : SPARSE AUTOENCODER (252 → 60 → 252)
# ============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.preprocessing import MinMaxScaler

# Reload data
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
X = df.drop(columns=['label', 'image_path']).values

# Normalize to [0,1] for sigmoid activation
mms = MinMaxScaler()
X_norm = mms.fit_transform(X)

# Build Sparse AE
input_dim = X.shape[1]  # 252
encoding_dim = 60

# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(encoding_dim, activation='sigmoid',
                       activity_regularizer=regularizers.l1(1e-5))(input_layer)
# Decoder
decoded = layers.Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = models.Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')  # MSE for reconstruction

# Custom sparsity is partly handled by activity_regularizer, which approximates KL penalty.
# For a proper sparsity constraint you would add a custom loss, but L1 activity
# regularizer encourages very low activations (close to 0), simulating sparsity.
# This is a practical simplification; final experiments can use a custom KL loss.

# Train
history = autoencoder.fit(X_norm, X_norm,
                          epochs=50,
                          batch_size=32,
                          shuffle=True,
                          validation_split=0.1,
                          verbose=0)
print("Sparse AE training complete. Last val loss:", history.history['val_loss'][-1])

# Extract encoder
encoder = models.Model(input_layer, encoded)
X_sparse_ae = encoder.predict(X_norm)

df_sae = pd.DataFrame(X_sparse_ae, columns=[f'SAE_{i+1}' for i in range(encoding_dim)])
df_sae['label'] = df['label'].values
sae_path = os.path.join(OUTPUT_DIR, 'reduced_sparse_ae_60.csv')
df_sae.to_csv(sae_path, index=False)
print(f"Sparse AE features saved to {sae_path}")

2026-05-01 18:24:03.569185: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Sparse AE training complete. Last val loss: 0.001629227539524436
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 984us/step
Sparse AE features saved to /kaggle/working/feature_analysis/reduced_sparse_ae_60.csv


In [10]:
# ============================================================================
# CELL 6 : STACKED AUTOENCODER (252 → 180 → 126 → 180 → 252)
# ============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models

# Reload data (already normalized from previous cell? Re-create to be safe)
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
X = df.drop(columns=['label', 'image_path']).values
mms = MinMaxScaler()
X_norm = mms.fit_transform(X)

input_dim = X.shape[1]
bottleneck_dim = 126

# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoded1 = layers.Dense(180, activation='relu')(input_layer)
encoded2 = layers.Dense(bottleneck_dim, activation='relu')(encoded1)

# Decoder
decoded1 = layers.Dense(180, activation='relu')(encoded2)
decoded2 = layers.Dense(input_dim, activation='sigmoid')(decoded1)

stacked_ae = models.Model(input_layer, decoded2)
stacked_ae.compile(optimizer='adam', loss='mse')

history = stacked_ae.fit(X_norm, X_norm,
                         epochs=50,
                         batch_size=32,
                         shuffle=True,
                         validation_split=0.1,
                         verbose=0)
print("Stacked AE training complete. Last val loss:", history.history['val_loss'][-1])

# Bottleneck encoder part
bottleneck_model = models.Model(input_layer, encoded2)
X_stacked_ae = bottleneck_model.predict(X_norm)

df_sae2 = pd.DataFrame(X_stacked_ae, columns=[f'StackedAE_{i+1}' for i in range(bottleneck_dim)])
df_sae2['label'] = df['label'].values
stacked_path = os.path.join(OUTPUT_DIR, 'reduced_stacked_ae_126.csv')
df_sae2.to_csv(stacked_path, index=False)
print(f"Stacked AE features saved to {stacked_path}")

Stacked AE training complete. Last val loss: 0.00010522972297621891
353/353 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Stacked AE features saved to /kaggle/working/feature_analysis/reduced_stacked_ae_126.csv


In [11]:
# ============================================================================
# CELL 7 : FEATURE SELECTION (ANOVA → 50, Chi2 → 40, RF → 50)
# ============================================================================
from sklearn.feature_selection import SelectKBest, f_classif, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import os

# Load data
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
X = df.drop(columns=['label', 'image_path']).values
y = df['label'].values
feature_names = df.drop(columns=['label', 'image_path']).columns

# ----- 1. ANOVA F-measure (select top 50) -----
selector_anova = SelectKBest(score_func=f_classif, k=50)
X_anova = selector_anova.fit_transform(X, y)
anova_mask = selector_anova.get_support()
anova_selected = feature_names[anova_mask].tolist()
print(f"ANOVA selected {len(anova_selected)} features")

# ----- 2. Chi-squared test (select top 40) -----
# Chi2 requires non-negative features, so we scale to [0,1]
mms = MinMaxScaler()
X_chi2 = mms.fit_transform(X)
selector_chi2 = SelectKBest(score_func=chi2, k=40)
X_chi2 = selector_chi2.fit_transform(X_chi2, y)
chi2_mask = selector_chi2.get_support()
chi2_selected = feature_names[chi2_mask].tolist()
print(f"Chi2 selected {len(chi2_selected)} features")

# ----- 3. Random Forest importance (select top 50) -----
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
importances = rf.feature_importances_
rf_indices = np.argsort(importances)[::-1][:50]   # top 50
rf_selected = feature_names[rf_indices].tolist()
print(f"RF selected {len(rf_selected)} features")

# Save the reduced datasets
df_anova = pd.DataFrame(X[:, anova_mask], columns=anova_selected)
df_anova['label'] = y
df_anova.to_csv(os.path.join(OUTPUT_DIR, 'selected_anova_50.csv'), index=False)

df_chi2 = pd.DataFrame(X[:, chi2_mask], columns=chi2_selected)
df_chi2['label'] = y
df_chi2.to_csv(os.path.join(OUTPUT_DIR, 'selected_chi2_40.csv'), index=False)

df_rf = pd.DataFrame(X[:, rf_indices], columns=rf_selected)
df_rf['label'] = y
df_rf.to_csv(os.path.join(OUTPUT_DIR, 'selected_rf_50.csv'), index=False)

print("All feature-selected datasets saved.")

ANOVA selected 50 features
Chi2 selected 40 features
RF selected 50 features
All feature-selected datasets saved.


In [14]:
# ============================================================================
# CELL : TRAIN & EVALUATE ANN ON ALL FEATURE SETS (FIXED)
# ============================================================================
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# ---------- CONFIG ----------
FEATURES_DIR = '/kaggle/working/feature_analysis'
RESULTS_DIR  = os.path.join(FEATURES_DIR, 'classification_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']

# Map method name → CSV filename
method_files = {
    'Original 252 Features': 'features_252_from_overlays.csv',
    'PCA (70)':              'reduced_pca_70.csv',
    'KPCA (65)':             'reduced_kpca_65.csv',
    'Sparse AE (60)':        'reduced_sparse_ae_60.csv',
    'Stacked AE (126)':      'reduced_stacked_ae_126.csv',
    'ANOVA F-measure (50)':  'selected_anova_50.csv',
    'Chi-square (40)':       'selected_chi2_40.csv',
    'Random Forest (50)':    'selected_rf_50.csv'
}

# ANN hyperparameters
EPOCHS = 100
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
RANDOM_STATE = 42

# ============================================================================
# Helper: build simple 2‑hidden‑layer MLP
# ============================================================================
def build_ann(input_dim, n_classes=4):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ============================================================================
# Loop over all methods
# ============================================================================
summary_results = {}

for method_name, csv_file in method_files.items():
    csv_path = os.path.join(FEATURES_DIR, csv_file)
    if not os.path.exists(csv_path):
        print(f"⚠️  {csv_file} not found. Skipping '{method_name}'.")
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {method_name}")
    print(f"{'='*60}")

    # 1. Load data
    df = pd.read_csv(csv_path)

    # Drop label and any non-numeric columns (like 'image_path')
    df_numeric = df.drop(columns=['label'], errors='ignore')
    # Also drop 'image_path' if it exists
    df_numeric = df_numeric.drop(columns=['image_path'], errors='ignore')
    # Optionally drop any other non-numeric columns (safety)
    df_numeric = df_numeric.select_dtypes(include=[np.number])

    X = df_numeric.values
    y = df['label'].values

    # 2. Train/validation split (stratified)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VALIDATION_SPLIT, stratify=y, random_state=RANDOM_STATE
    )

    # 3. Standardize
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    # 4. One‑hot encode labels
    y_train_cat = to_categorical(y_train, num_classes=4)
    y_val_cat   = to_categorical(y_val, num_classes=4)

    # 5. Build and train model
    model = build_ann(input_dim=X_train.shape[1])
    early_stop = EarlyStopping(monitor='val_accuracy', patience=20,
                               restore_best_weights=True, verbose=0)

    history = model.fit(
        X_train, y_train_cat,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val_cat),
        callbacks=[early_stop],
        verbose=0
    )

    # 6. Evaluate on validation set
    y_pred_probs = model.predict(X_val)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # 7. Classification report (save as text)
    report = classification_report(y_val, y_pred, target_names=CLASS_NAMES, digits=4)
    print(report)

    # Save report to file
    report_path = os.path.join(RESULTS_DIR, f'{method_name.replace(" ", "_")}_report.txt')
    with open(report_path, 'w') as f:
        f.write(report)

    # 8. Confusion matrix (save figure)
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Confusion Matrix - {method_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, f'{method_name.replace(" ", "_")}_cm.png')
    plt.savefig(cm_path, dpi=150)
    plt.close()

    # 9. Training curves (accuracy & loss)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title(f'Accuracy - {method_name}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    # Loss
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'Loss - {method_name}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    plt.tight_layout()
    curves_path = os.path.join(RESULTS_DIR, f'{method_name.replace(" ", "_")}_curves.png')
    plt.savefig(curves_path, dpi=150)
    plt.close()

    # 10. Store metrics for summary
    val_acc = max(history.history['val_accuracy'])
    val_loss = min(history.history['val_loss'])
    summary_results[method_name] = {
        'val_accuracy': val_acc,
        'val_loss': val_loss,
        'report_path': report_path,
        'cm_path': cm_path,
        'curves_path': curves_path
    }

# 11. Save summary as JSON for easy comparison
summary_path = os.path.join(RESULTS_DIR, 'summary_metrics.json')
with open(summary_path, 'w') as f:
    json.dump({k: {'val_accuracy': v['val_accuracy'], 'val_loss': v['val_loss']}
               for k, v in summary_results.items()}, f, indent=4)

print(f"\n✅ All experiments completed. Results saved to: {RESULTS_DIR}")
print("Summary metrics:\n", summary_path)


Processing: Original 252 Features
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
                      precision    recall  f1-score   support

             Healthy     0.8687    0.8996    0.8839       478
              Yellow     0.8920    0.8443    0.8675       636
Cercospora Leaf Spot     0.8773    0.8348    0.8555       454
 Bacterial Leaf Spot     0.8303    0.8776    0.8533       686

            accuracy                         0.8642      2254
           macro avg     0.8671    0.8641    0.8651      2254
        weighted avg     0.8653    0.8642    0.8642      2254


Processing: PCA (70)
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
                      precision    recall  f1-score   support

             Healthy     0.8740    0.8996    0.8866       478
              Yellow     0.8898    0.8129    0.8496       636
Cercospora Leaf Spot     0.8698    0.8678    0.8688       454
 Bacterial Leaf Spot     0.8173    0.8673    0.8416       686

            accuracy                         0.8589 

In [15]:
# ============================================================================
# CELL: ZIP THE FEATURE ANALYSIS FOLDER FOR DOWNLOAD
# ============================================================================
import shutil
import os

FEATURES_DIR = '/kaggle/working/feature_analysis'  # folder to zip
ZIP_PATH     = '/kaggle/working/feature_analysis.zip'  # output zip file

# Create zip archive
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', FEATURES_DIR)

print(f"✅ Done! Download the zip from:\n{ZIP_PATH}")

✅ Done! Download the zip from:
/kaggle/working/feature_analysis.zip


## CNN and XAI

In [1]:
# ============================================================================
# CELL: TRAIN CNN ON SEGMENTED OVERLAY IMAGES (for XAI)
# ============================================================================
import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# ---------- CONFIG ----------
IMAGE_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
IMG_SIZE  = 256
BATCH     = 32
EPOCHS    = 30
SEED      = 42
CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']

# ---------- Load image paths and labels ----------
paths, labels = [], []
for idx, cls in enumerate(CLASS_NAMES):
    for fname in os.listdir(IMAGE_DIR):
        if fname.startswith(cls + '__'):
            paths.append(os.path.join(IMAGE_DIR, fname))
            labels.append(idx)

X_paths, y = np.array(paths), np.array(labels)

# Shuffle and split (keeping 20% for validation)
train_paths, val_paths, y_train, y_val = train_test_split(
    X_paths, y, test_size=0.2, stratify=y, random_state=SEED
)

# ---------- Keras data generators ----------
def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)  # overlay saved as PNG
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, y_train))
train_ds = train_ds.shuffle(1000).map(lambda p, l: (load_img(p), l),
                                      num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, y_val))
val_ds = val_ds.map(lambda p, l: (load_img(p), l),
                    num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

# ---------- Simple CNN model ----------
def create_cnn():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(4, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = create_cnn()
cnn_model.summary()

# ---------- Train ----------
early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                        callbacks=[early_stop], verbose=1)

# ---------- Save model ----------
cnn_model.save('/kaggle/working/cnn_overlay_model.h5')
print("CNN model saved to /kaggle/working/cnn_overlay_model.h5")

2026-05-01 19:04:29.902561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777662270.102982      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777662270.167019      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777662270.640900      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777662270.640943      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777662270.640947      57 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 249,348 (974.02 KB)

 Trainable params: 249,348 (974.02 KB)

...

I0000 00:00:1777662305.242013     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


281/282 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.4927 - loss: 0.9392

2026-05-01 19:05:24.510661: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-01 19:05:24.677200: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


282/282 ━━━━━━━━━━━━━━━━━━━━ 37s 100ms/step - accuracy: 0.4931 - loss: 0.9384 - val_accuracy: 0.6894 - val_loss: 0.6532
Epoch 2/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 19s 67ms/step - accuracy: 0.6458 - loss: 0.7151 - val_accuracy: 0.6681 - val_loss: 0.6671
Epoch 3/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 19s 68ms/step - accuracy: 0.6632 - loss: 0.7003 - val_accuracy: 0.6597 - val_loss: 0.6713
Epoch 4/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 20s 70ms/step - accuracy: 0.6911 - loss: 0.6798 - val_accuracy: 0.7209 - val_loss: 0.6208
Epoch 5/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - accuracy: 0.7113 - loss: 0.6356 - val_accuracy: 0.6761 - val_loss: 0.6578
Epoch 6/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 73ms/step - accuracy: 0.7250 - loss: 0.6143 - val_accuracy: 0.7427 - val_loss: 0.6045
Epoch 7/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 73ms/step - accuracy: 0.7400 - loss: 0.6009 - val_accuracy: 0.7076 - val_loss: 0.6497
Epoch 8/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 20s 72ms/step - accuracy: 0.7307 - loss: 0.6059 - val_accura

CNN model saved to /kaggle/working/cnn_overlay_model.h5


In [2]:
# ============================================================================
# CELL: INSTALL XAI LIBRARIES (run once)
# ============================================================================
!pip install tf-keras-vis shap lime --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 2.2 MB/s eta 0:00:00


In [12]:
# ============================================================================
# CELL: XAI FOR CNN (Grad-CAM, Grad-CAM++, Score-CAM, LIME, SHAP)
# ============================================================================
import os, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import skimage.segmentation

# XAI imports
from tf_keras_vis.utils.scores import CategoricalScore
from tf_keras_vis.gradcam import Gradcam, GradcamPlusPlus
from tf_keras_vis.scorecam import Scorecam
import shap
import lime
from lime import lime_image

# ---------- Config ----------
MODEL_PATH = '/kaggle/working/cnn_overlay_model.h5'
OVERLAY_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
OUTPUT_XAI_DIR = '/kaggle/working/xai_cnn_results'
os.makedirs(OUTPUT_XAI_DIR, exist_ok=True)

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
IMG_SIZE = 256

# ---------- Load model and force build ----------
model = load_model(MODEL_PATH, compile=True)
# Force build by calling model on a dummy input
_ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))   # builds the graph
print("Model built successfully")
model.summary()

# Find the last Dense layer with 4 units (logits)
logits_layer = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Dense) and layer.units == 4:
        logits_layer = layer
        break
if logits_layer is None:
    raise ValueError("Could not find a Dense layer with 4 units (logits). Check model architecture.")

# Create logits model - use model.inputs[0] (now defined)
logits_model = tf.keras.Model(inputs=model.inputs[0], outputs=logits_layer.output)
print(f"Logits model created using layer: {logits_layer.name}")

# ---------- Prepare one image per class ----------
def load_preprocess_img(path):
    img = image.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    return img

# Collect one image per class (files named like "Healthy__xxx.jpg")
sample_paths = []
for cls_id, cls_name in enumerate(CLASS_NAMES):
    candidates = [f for f in os.listdir(OVERLAY_DIR) if f.startswith(cls_name + '__')]
    if not candidates:
        print(f"Warning: No image found for class {cls_name}")
        continue
    chosen = random.choice(candidates)
    sample_paths.append((cls_id, os.path.join(OVERLAY_DIR, chosen), cls_name))

if not sample_paths:
    raise FileNotFoundError("No sample images found for any class in OVERLAY_DIR")

# ---------- tf-keras-vis setup ----------
def model_modifier(m):
    m.layers[-1].activation = tf.keras.activations.linear
    return m

gradcam = Gradcam(model, model_modifier=model_modifier, clone=True)
gradcam_pp = GradcamPlusPlus(model, model_modifier=model_modifier, clone=True)
scorecam = Scorecam(model, model_modifier=None)

# ---------- SHAP background (20 random images from overlays) ----------
background = []
image_paths = [os.path.join(OVERLAY_DIR, f) for f in os.listdir(OVERLAY_DIR) 
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
if len(image_paths) == 0:
    raise FileNotFoundError(f"No image files found in {OVERLAY_DIR}")

random.shuffle(image_paths)
background_paths = image_paths[:20]

for path in background_paths:
    img = image.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    background.append(img)

background = np.array(background, dtype=np.float32)
print(f"✅ Loaded {len(background)} background images for SHAP")

# ---------- Loop over sample images ----------
for cls_id, img_path, cls_name in sample_paths:
    print(f"\nProcessing {cls_name} ...")
    img = load_preprocess_img(img_path)
    img_batch = np.expand_dims(img, axis=0)

    # Prediction
    pred = model.predict(img_batch, verbose=0)[0]
    pred_class = np.argmax(pred)
    target_class = pred_class
    print(f"  True class: {cls_name} | Predicted: {CLASS_NAMES[pred_class]} (confidence {pred[pred_class]:.3f})")

    # Prepare score for GradCAM etc.
    score = CategoricalScore(target_class)

    # ----- Grad-CAM -----
    try:
        cam = gradcam(score, img_batch, penultimate_layer=-1)
        cam = np.squeeze(cam)
        _, ax = plt.subplots(1,3, figsize=(15,5))
        ax[0].imshow(img)
        ax[0].set_title(f"{cls_name} (pred: {CLASS_NAMES[pred_class]})")
        ax[1].imshow(cam, cmap='jet')
        ax[1].set_title("Grad-CAM")
        ax[2].imshow(img)
        ax[2].imshow(cam, cmap='jet', alpha=0.4)
        ax[2].set_title("Overlay")
        for a in ax: a.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_gradcam.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  Grad-CAM failed: {e}")

    # ----- Grad-CAM++ -----
    try:
        cam_pp = gradcam_pp(score, img_batch, penultimate_layer=-1)
        cam_pp = np.squeeze(cam_pp)
        _, ax = plt.subplots(1,3, figsize=(15,5))
        ax[0].imshow(img)
        ax[0].set_title(f"{cls_name} original")
        ax[1].imshow(cam_pp, cmap='jet')
        ax[1].set_title("Grad-CAM++")
        ax[2].imshow(img)
        ax[2].imshow(cam_pp, cmap='jet', alpha=0.4)
        ax[2].set_title("Overlay")
        for a in ax: a.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_gradcampp.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  Grad-CAM++ failed: {e}")

    # ----- Score-CAM -----
    try:
        sc = scorecam(score, img_batch, penultimate_layer=-1)
        sc = np.squeeze(sc)
        _, ax = plt.subplots(1,3, figsize=(15,5))
        ax[0].imshow(img)
        ax[0].set_title("Original")
        ax[1].imshow(sc, cmap='jet')
        ax[1].set_title("Score-CAM")
        ax[2].imshow(img)
        ax[2].imshow(sc, cmap='jet', alpha=0.4)
        ax[2].set_title("Overlay")
        for a in ax: a.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_scorecam.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  Score-CAM failed: {e}")

    # ----- LIME -----
    try:
        explainer = lime_image.LimeImageExplainer()
        def predict_fn(imgs):
            imgs = imgs.astype(np.float32) / 255.0
            return model.predict(imgs, verbose=0)
        explanation = explainer.explain_instance(
            (img * 255).astype(np.uint8),
            classifier_fn=predict_fn,
            top_labels=1,
            hide_color=0,
            num_samples=500
        )
        temp, mask = explanation.get_image_and_mask(
            target_class,
            positive_only=True,
            num_features=5,
            hide_rest=False
        )
        plt.figure(figsize=(8,8))
        plt.imshow(temp / 255.0)
        plt.title(f"LIME explanation for {cls_name}")
        plt.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_lime.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  LIME failed: {e}")

    # ----- SHAP (using logits model) -----
    try:
        e = shap.DeepExplainer(logits_model, background)
        shap_values = e.shap_values(img_batch)   # list of arrays, one per class
        # shap_values length = number of classes (4)
        if len(shap_values) != len(CLASS_NAMES):
            print(f"  Warning: SHAP returned {len(shap_values)} value sets, expected {len(CLASS_NAMES)}")
        # Visualise for the predicted class
        class_idx = target_class
        if class_idx < len(shap_values):
            shap_arr = shap_values[class_idx]        # shape (1, 256, 256, 3)
            shap_heatmap = np.abs(shap_arr[0]).max(axis=-1)   # (256,256)
            plt.figure(figsize=(8,8))
            plt.imshow(img)
            plt.imshow(shap_heatmap, cmap='inferno', alpha=0.5)
            plt.title(f"SHAP heatmap (logits) for {cls_name} (class {CLASS_NAMES[class_idx]})")
            plt.axis('off')
            plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_shap.png'), bbox_inches='tight')
            plt.close()
        else:
            print(f"  SHAP index {class_idx} out of range")
    except Exception as e:
        print(f"  SHAP failed: {e}")

print("\n✅ All CNN XAI visualizations saved to:", OUTPUT_XAI_DIR)

Model built successfully


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 249,350 (974.03 KB)

 Trainable params: 249,348 (974.02 KB)

...

  SHAP index 2 out of range

Processing Bacterial Leaf Spot ...
  True class: Bacterial Leaf Spot | Predicted: Bacterial Leaf Spot (confidence 0.944)
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


  0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/shap/explainers/_deep/deep_tf.py:94: UserWarning: Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(20, 256, 256, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(40, 256, 256, 3))']
  warnings.warn(msg)


  SHAP index 3 out of range

✅ All CNN XAI visualizations saved to: /kaggle/working/xai_cnn_results


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(1, 256, 256, 3))']
  warnings.warn(msg)


In [13]:
# ============================================================================
# CELL: ZIP THE FEATURE ANALYSIS FOLDER FOR DOWNLOAD
# ============================================================================
import shutil
import os

FEATURES_DIR = '/kaggle/working/xai_cnn_results'  # folder to zip
ZIP_PATH     = '/kaggle/working/xai_cnn_results.zip'  # output zip file

# Create zip archive
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', FEATURES_DIR)

print(f"✅ Done! Download the zip from:\n{ZIP_PATH}")

✅ Done! Download the zip from:
/kaggle/working/xai_cnn_results.zip
